# Part 1 – Exploratory Data Analysis

Chicken egg production for the configured flock user.

This notebook mirrors `src/run_eda.py`. Prefer running the script to regenerate `reports/`:

```bash
python -m src.build_daily_dataset
python -m src.run_eda
```

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data_processing import prepare_daily, aggregate_weekly, save_processed
from src.features import flag_anomalies_zscore, event_impact_windows

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## Load & prepare

In [ ]:
df = prepare_daily()
df = flag_anomalies_zscore(df, value_col="eggs_per_hen")
save_processed(df)
weekly = aggregate_weekly(df)

print(df[["date", "eggs", "hens", "feed_kg", "temp_avg", "eggs_per_hen"]].head())
df.describe(include="number").T

## Production time series

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(df["date"], df["eggs"]); axes[0].set_ylabel("Eggs")
axes[1].plot(df["date"], df["eggs_per_hen"]); axes[1].set_ylabel("Eggs/hen")
axes[2].plot(df["date"], df["feed_kg"]); axes[2].set_ylabel("Feed kg")
plt.show()

## Seasonality

In [ ]:
sns.boxplot(data=df, x="month", y="eggs")
plt.title("Eggs by month")
plt.show()

df.groupby("month")["eggs_per_hen"].mean().plot(kind="bar", title="Mean eggs/hen by month")
plt.show()

## Weather impact

In [ ]:
plot_df = df.dropna(subset=["eggs_per_hen", "temp_avg"])
sns.scatterplot(data=plot_df, x="temp_avg", y="eggs_per_hen", alpha=0.4)
plt.title("Temp avg vs eggs/hen")
plt.show()

cols = ["eggs_per_hen", "feed_kg", "temp_avg", "precipitation", "daylight_minutes", "hens"]
sns.heatmap(df[cols].corr(), annot=True, cmap="RdBu_r", center=0)
plt.show()

## Events & anomalies

In [ ]:
for col in ["event_death", "event_feed_change", "event_added"]:
    n = int(df[col].sum())
    print(col, n)
    if n == 0:
        continue
    win = event_impact_windows(df, col)
    curve = win.groupby("day_offset")["eggs_per_hen"].mean()
    curve.plot(label=col)
plt.axvline(0, color="k", ls="--")
plt.legend(); plt.title("Event impact"); plt.show()

anom = df[df["is_anomaly_low"] | df["is_anomaly_high"]]
plt.plot(df["date"], df["eggs_per_hen"], alpha=0.5)
plt.scatter(anom["date"], anom["eggs_per_hen"], c="red", s=20)
plt.title("Anomalies"); plt.show()

## Written summary

In [ ]:
summary = (ROOT / "reports" / "eda_summary.md")
if summary.exists():
    display(Markdown(summary.read_text(encoding="utf-8")))
else:
    print("Run: python -m src.run_eda")